In [16]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import countDistinct
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("CaseStudy") \
    .getOrCreate()

In [2]:
customers = spark.read.csv("casestudy/customers.csv", header=True, inferSchema=True)

orders = spark.read.csv("casestudy/orders.csv", header=True, inferSchema=True)

order_items = spark.read.csv("casestudy/order_items.csv", header=True, inferSchema=True)

products = spark.read.csv("casestudy/products.csv", header=True, inferSchema=True)

returns = spark.read.csv("casestudy/returns.csv", header=True, inferSchema=True)

In [3]:
customers.show(5)
orders.show(5)

+-----------+-------------+--------+-----+-----------------+----------------+
|customer_id|customer_name|    city|state|registration_date|customer_segment|
+-----------+-------------+--------+-----+-----------------+----------------+
|          1|   Customer_1|Columbus|   OH|       2023-10-17|             VIP|
|          2|   Customer_2|   Miami|   CA|       2022-04-25|         Premium|
|          3|   Customer_3| Atlanta|   FL|       2022-01-26|         Premium|
|          4|   Customer_4| Chicago|   OH|       2022-10-09|        Standard|
|          5|   Customer_5|Columbus|   IL|       2022-09-08|         Premium|
+-----------+-------------+--------+-----+-----------------+----------------+
only showing top 5 rows
+--------+-----------+----------+------------+------------+
|order_id|customer_id|order_date|payment_mode|order_status|
+--------+-----------+----------+------------+------------+
|       1|      54630|2024-01-25| Credit Card|   Delivered|
|       2|      22415|2024-07-08| 

In [4]:
#Q1
print("Customers:", customers.count())
print("Products:", products.count())
print("Orders:", orders.count())
print("Returned Orders:", returns.count())

Customers: 100000
Products: 50000
Orders: 1000000
Returned Orders: 100000


In [7]:
#Q2-JOIN
sales_df = order_items.join(
    products,
    "product_id"
)

sales_df.show(5)

+----------+-------------+--------+--------+-------------+-------------+-----------+-------+---------+
|product_id|order_item_id|order_id|quantity|selling_price| product_name|   category|  brand|unit_cost|
+----------+-------------+--------+--------+-------------+-------------+-----------+-------+---------+
|     28849|            1|  227444|       5|       727.98|Product_28849|       Toys|Brand_D|   547.15|
|     25471|            2|   32708|       2|       203.02|Product_25471|Electronics|Brand_D|   180.83|
|      2818|            3|  426242|       5|      1061.81| Product_2818|     Beauty|Brand_A|   708.47|
|     35931|            4|  236965|       5|      1005.49|Product_35931|       Toys|Brand_D|   672.49|
|     48310|            5|  289552|       4|       540.78|Product_48310|   Clothing|Brand_E|   321.27|
+----------+-------------+--------+--------+-------------+-------------+-----------+-------+---------+
only showing top 5 rows


In [8]:
#Q2- calculate 

sales_by_category = sales_df.groupBy("category") \
    .agg(
        round(
            sum(col("quantity") * col("selling_price")),
            2
        ).alias("total_sales")
    )

sales_by_category.show()

[Stage 33:=============================>                            (1 + 1) / 2]

+--------------+--------------+
|      category|   total_sales|
+--------------+--------------+
|Home & Kitchen|7.5813887328E8|
|        Sports|7.4333886813E8|
|   Electronics|7.4426650411E8|
|      Clothing|7.4192279457E8|
|         Books|7.4649077835E8|
|        Beauty| 7.626693059E8|
|          Toys| 7.446190723E8|
+--------------+--------------+



In [10]:
#Q.3 JOIN-EVERYTHING FIRST

cust_sales = customers \
    .join(orders, "customer_id") \
    .join(order_items, "order_id")

#And then calculate

top_customers = cust_sales.groupBy(
    "customer_id",
    "customer_name"
).agg(
    round(
        sum(
            col("quantity") * col("selling_price")
        ),
        2
    ).alias("purchase_amount")
).orderBy(
    desc("purchase_amount")
).limit(10)

top_customers.show()

[Stage 41:=============================>                            (1 + 1) / 2]

+-----------+--------------+---------------+
|customer_id| customer_name|purchase_amount|
+-----------+--------------+---------------+
|      93094|Customer_93094|      181569.68|
|      64560|Customer_64560|       169060.4|
|      23289|Customer_23289|       161573.8|
|      52275|Customer_52275|      153364.79|
|      61218|Customer_61218|      153067.55|
|      52034|Customer_52034|      152680.05|
|      40442|Customer_40442|      151037.32|
|      60528|Customer_60528|      148691.95|
|      84830|Customer_84830|      148363.84|
|      82593|Customer_82593|      148281.04|
+-----------+--------------+---------------+



In [11]:
#Q.4 JOIN FIRST
sales_trend = orders \
    .join(order_items, "order_id")

#CREATE MONTH COLUMN
sales_trend = sales_trend.withColumn(
    "month",
    date_format(
        col("order_date"),
        "yyyy-MM"
    )
)

#calculate
monthly_sales = sales_trend.groupBy(
    "month"
).agg(
    round(
        sum(
            col("quantity") * col("selling_price")
        ),
        2
    ).alias("sales")
).orderBy("month")

monthly_sales.show()

[Stage 50:=============================>                            (1 + 1) / 2]

+-------+--------------+
|  month|         sales|
+-------+--------------+
|2024-01|4.4457777576E8|
|2024-02| 4.153661442E8|
|2024-03|4.4362824541E8|
|2024-04|4.2782097434E8|
|2024-05|4.4481061895E8|
|2024-06|4.3170515406E8|
|2024-07|4.4367051912E8|
|2024-08|4.4109517702E8|
|2024-09|4.3107152608E8|
|2024-10|4.4136378931E8|
|2024-11|4.3362336404E8|
|2024-12|4.4271290835E8|
+-------+--------------+



In [24]:
#Q.5 TOTAL PRODUCTS(total orders)

total_orders = products \
    .join(order_items, "product_id")

#RETURNED ORDER

returned_orders = products \
    .join(order_items, "product_id") \
    .join(returns, "order_id")

#COUNTS

from pyspark.sql import functions as F
total_count = total_orders.groupBy(
    "category"
).agg(
    countDistinct("order_id").alias("total_orders")
)

return_count = returned_orders.groupBy(
    "category"
).agg(
    countDistinct("order_id").alias("returned_orders")
)

#THEN I CALCULATE PRECENTAGE 

return_percentage = total_count.join(
    return_count,
    "category",
    "left"
).fillna(0)

return_percentage = return_percentage.withColumn(
    "return_percentage",
    round(
        col("returned_orders")
        /
        col("total_orders")
        * 100,
        2
    )
)

return_percentage.show()

[Stage 127:>                                                        (0 + 2) / 2]

+--------------+------------+---------------+-----------------+
|      category|total_orders|returned_orders|return_percentage|
+--------------+------------+---------------+-----------------+
|Home & Kitchen|      351594|          35260|            10.03|
|        Sports|      345930|          34699|            10.03|
|   Electronics|      346773|          34744|            10.02|
|      Clothing|      347805|          34680|             9.97|
|         Books|      347520|          34832|            10.02|
|        Beauty|      349992|          35070|            10.02|
|          Toys|      349584|          35112|            10.04|
+--------------+------------+---------------+-----------------+



In [19]:
#Q.6 JOIN
payment_df = customers \
    .join(orders, "customer_id")

#COUNT
payment_count = payment_df.groupBy(
    "state",
    "payment_mode"
).count()

#WINDOW
window_spec = Window.partitionBy(
    "state"
).orderBy(
    desc("count")
)

#THEN I CHECK RANK
preferred_payment = payment_count \
    .withColumn(
        "rank",
        row_number().over(window_spec)
    ) \
    .filter(
        col("rank") == 1
    )

preferred_payment.show()

[Stage 71:=============================>                            (1 + 1) / 2]

+-----+----------------+-----+----+
|state|    payment_mode|count|rank|
+-----+----------------+-----+----+
|   CA|             UPI|20246|   1|
|   FL|      Debit Card|20010|   1|
|   GA|     Net Banking|20041|   1|
|   IL|Cash on Delivery|20498|   1|
|   MI|      Debit Card|20416|   1|
|   NC|     Net Banking|19596|   1|
|   NY|      Debit Card|20369|   1|
|   OH|     Net Banking|20351|   1|
|   TX|             UPI|20065|   1|
|   WA|             UPI|20244|   1|
+-----+----------------+-----+----+



In [20]:
#Q.7 JOIN

customer_analysis = customers \
    .join(orders, "customer_id") \
    .join(order_items, "order_id") \
    .join(products, "product_id")

#CALCULATE 

qualified_customers = customer_analysis.groupBy(
    "customer_id",
    "customer_name"
).agg(
    countDistinct(
        "category"
    ).alias(
        "category_count"
    ),

    round(
        sum(
            col("quantity")
            *
            col("selling_price")
        ),
        2
    ).alias(
        "total_spent"
    )
)

#THEN NOW I APPLY FILTER IN IT

qualified_customers.filter(
    (col("category_count") >= 5)
    &
    (col("total_spent") > 100000)
).show()

[Stage 87:=============================>                            (1 + 1) / 2]

+-----------+--------------+--------------+-----------+
|customer_id| customer_name|category_count|total_spent|
+-----------+--------------+--------------+-----------+
|      26241|Customer_26241|             7|   121047.8|
|      52297|Customer_52297|             7|  107812.68|
|      41157|Customer_41157|             7|   105187.1|
|      84144|Customer_84144|             7|  103326.02|
|      46860|Customer_46860|             7|  100298.43|
|      81900|Customer_81900|             7|  127465.68|
|      17271|Customer_17271|             7|  103096.35|
|      97920|Customer_97920|             7|  117591.17|
|      90203|Customer_90203|             7|  109729.63|
|      18149|Customer_18149|             7|   101780.7|
|      46060|Customer_46060|             7|  115805.04|
|      90970|Customer_90970|             7|  111172.63|
|      17979|Customer_17979|             7|  132436.51|
|      60321|Customer_60321|             7|  108877.45|
|      60430|Customer_60430|             7|  100

In [22]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Join products and order_items

revenue_df = products.join(
    order_items,
    "product_id"
)

# Calculate revenue for each product

product_revenue = revenue_df.groupBy(
    "category",
    "product_id",
    "product_name"
).agg(
    round(
        sum(
            col("quantity") * col("selling_price")
        ),
        2
    ).alias("revenue")
)

print("Product Revenue")

product_revenue.show(truncate=False)

# Create Window Specification

window_spec = Window.partitionBy(
    "category"
).orderBy(
    desc("revenue")
)

# Find Top 3 Products in each Category

top_products = product_revenue.withColumn(
    "rank",
    row_number().over(window_spec)
)

top_products = top_products.filter(
    col("rank") <= 3
)

print("Top 3 Products by Revenue within each Category")

top_products.orderBy(
    "category",
    "rank"
).show(truncate=False)

Product Revenue


+-----------+----------+-------------+---------+
|category   |product_id|product_name |revenue  |
+-----------+----------+-------------+---------+
|Clothing   |1280      |Product_1280 |185289.69|
|Electronics|24507     |Product_24507|90113.4  |
|Electronics|37498     |Product_37498|53721.08 |
|Beauty     |18640     |Product_18640|107917.53|
|Sports     |45205     |Product_45205|114789.16|
|Beauty     |1893      |Product_1893 |18459.02 |
|Electronics|17156     |Product_17156|158318.22|
|Toys       |44426     |Product_44426|101340.69|
|Electronics|21030     |Product_21030|105387.99|
|Electronics|49242     |Product_49242|72795.13 |
|Sports     |11448     |Product_11448|74467.66 |
|Sports     |1300      |Product_1300 |164779.81|
|Toys       |37099     |Product_37099|83755.54 |
|Clothing   |15493     |Product_15493|101753.73|
|Electronics|1979      |Product_1979 |172043.74|
|Electronics|33579     |Product_33579|152319.54|
|Toys       |45786     |Product_45786|128603.1 |
|Beauty     |42199  

[Stage 100:============================>                            (1 + 1) / 2]

+--------------+----------+-------------+---------+----+
|category      |product_id|product_name |revenue  |rank|
+--------------+----------+-------------+---------+----+
|Beauty        |44016     |Product_44016|277567.99|1   |
|Beauty        |14849     |Product_14849|274894.2 |2   |
|Beauty        |786       |Product_786  |272174.7 |3   |
|Books         |35314     |Product_35314|296468.78|1   |
|Books         |28311     |Product_28311|286757.72|2   |
|Books         |37479     |Product_37479|276736.71|3   |
|Clothing      |7025      |Product_7025 |293821.97|1   |
|Clothing      |1560      |Product_1560 |288474.09|2   |
|Clothing      |31322     |Product_31322|282241.17|3   |
|Electronics   |6719      |Product_6719 |299113.87|1   |
|Electronics   |23519     |Product_23519|289561.72|2   |
|Electronics   |38170     |Product_38170|288875.23|3   |
|Home & Kitchen|5012      |Product_5012 |305836.22|1   |
|Home & Kitchen|37452     |Product_37452|286817.42|2   |
|Home & Kitchen|27682     |Prod